# PHASE 0: Environment Setup & Safe Extraction

### PHASE 0.A: ENVIRONMENT SETUP

In [1]:
import sys
import os

# 1. Add the project root to the Python path to import 'src'
project_root = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
if project_root not in sys.path:
    sys.path.append(project_root)

# 2. Import project configuration (Absolute paths)
from src import config

# 3. Import Data Science libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 4. Set visual style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("✅ Setup complete. Ready to load data.")

✅ Setup complete. Ready to load data.


### PHASE 0.B: DATA EXTRACTION

In [2]:

print("Loading Open Data INE (Instituto Nacional de Estadística)... This might take a few seconds.\n")

try:
    # Load the INE dataset
    df_inerent = pd.read_csv(
        config.PATH_INERENT, 
        sep=';',
        encoding='latin-1', # --> To read ñ's and accents correctly
        low_memory=False 
    )
    
    # Print the size of the dataset
    total_rows = df_inerent.shape[0]
    total_cols = df_inerent.shape[1]
    
    print(f"✅ Data loaded successfully!")
    print(f"📊 Dataset size: {total_rows:,} rows and {total_cols} columns.")
    
    # Show the first 5 rows to verify it loaded correctly
    display(df_inerent.head())
    
except Exception as e:
    print(f"❌ Error loading data: {e}")

Loading Open Data INE (Instituto Nacional de Estadística)... This might take a few seconds.

✅ Data loaded successfully!
📊 Dataset size: 242,622 rows and 6 columns.


,ï»¿Municipios,Distritos,Secciones,Indicadores de renta media y mediana,Periodo,Total
0,08001 Abrera,NaN,NaN,Renta neta media por persona,2023,16.682
1,08001 Abrera,NaN,NaN,Renta neta media por persona,2022,15.746
2,08001 Abrera,NaN,NaN,Renta neta media por persona,2021,15.134
3,08001 Abrera,NaN,NaN,Renta neta media por persona,2020,14.455
4,08001 Abrera,NaN,NaN,Renta neta media por persona,2019,14.347


# PHASE 1: DATA CLEANING & GEOGRAPHIC FILTERING

In [3]:
print("🧹 Cleaning column names and filtering for Barcelona...\n")

# 1. Rename columns to fix the BOM (Byte Order Mark) character and simplify long names
df_inerent.rename(columns={
    df_inerent.columns[0]: 'Municipio',  
    'Indicadores de renta media y mediana': 'Indicador',
    'Total': 'Renta_Media'
}, inplace=True)

# 2. Filter ONLY for Barcelona municipality (INE code for Barcelona is '08019')
df_bcn_renta = df_inerent[df_inerent['Municipio'].str.contains('08019', na=False)].copy()

# 3. Filter for the specific metric we want: 'Renta neta media por persona'
df_bcn_renta = df_bcn_renta[df_bcn_renta['Indicador'] == 'Renta neta media por persona']

# 4. Find the latest available year dynamically and filter for it
latest_year = df_bcn_renta['Periodo'].max()
df_bcn_renta = df_bcn_renta[df_bcn_renta['Periodo'] == latest_year]

# 5. Find missing values in Barcelona
print("🔍 Auditing Missing Values (Nulls) in Barcelona data:")
# Calculate total nulls per column
null_counts = df_bcn_renta.isnull().sum()
# Display only columns that actually have missing values
missing_data = null_counts[null_counts > 0]

if missing_data.empty:
    print("✅ No missing values in this dataset.")
else:
    print("⚠️ Missing values detected:")
    print(missing_data)
print("=" * 50)

print(f"\n✅ Success! Filtered down from 242,622 to {len(df_bcn_renta)} rows.")
print(f"📊 Showing Data for Barcelona (Year: {latest_year}).")

display(df_bcn_renta.head())

🧹 Cleaning column names and filtering for Barcelona...

🔍 Auditing Missing Values (Nulls) in Barcelona data:
⚠️ Missing values detected:
Distritos     1
Secciones    11
dtype: int64

✅ Success! Filtered down from 242,622 to 1079 rows.
📊 Showing Data for Barcelona (Year: 2023).


,Municipio,Distritos,Secciones,Indicador,Periodo,Renta_Media
13986,08019 Barcelona,NaN,NaN,Renta neta media por persona,2023,19.527
14040,08019 Barcelona,0801901 Barcelona distrito 01,NaN,Renta neta media por persona,2023,13.990
14094,08019 Barcelona,0801901 Barcelona distrito 01,0801901001 Barcelona secciÃ³n 01001,Renta neta media por persona,2023,13.122
14148,08019 Barcelona,0801901 Barcelona distrito 01,0801901002 Barcelona secciÃ³n 01002,Renta neta media por persona,2023,10.966
14202,08019 Barcelona,0801901 Barcelona distrito 01,0801901003 Barcelona secciÃ³n 01003,Renta neta media por persona,2023,10.702


# PHASE 2: EXTRACTING DISTRICT-LEVEL INCOME

In [4]:
print("🎯 Isolating District-level data and extracting District Codes...\n")

# 1. Filter: We want rows that DO have a District (notna) but DO NOT have a Section (isna)
df_renta_distritos = df_bcn_renta[
    df_bcn_renta['Distritos'].notna() & 
    df_bcn_renta['Secciones'].isna()
].copy()

# 2. Extract the district number (e.g., from "0801901 Barcelona distrito 01" we get "1")
# Split the text by spaces and keep the last segment
df_renta_distritos['Codi_Districte'] = df_renta_distritos['Distritos'].str.split(' ').str[-1].astype(int)

# 3. Clean the Income (remove the thousands separator dot and convert to an integer)
df_renta_distritos['Renta_Media'] = df_renta_distritos['Renta_Media'].astype(str).str.replace('.', '').astype(int)

# 4. Keep only the relevant columns, sort by district code, and reset the index
df_renta_distritos = df_renta_distritos[['Codi_Districte', 'Renta_Media']].sort_values('Codi_Districte').reset_index(drop=True)

print(f"✅ Success! Isolated {len(df_renta_distritos)} districts.")
display(df_renta_distritos)

🎯 Isolating District-level data and extracting District Codes...

✅ Success! Isolated 10 districts.


,Codi_Districte,Renta_Media
0,1,13990
1,2,21976
2,3,16911
3,4,26069
4,5,31710
5,6,21251
6,7,17234
7,8,13964
8,9,16979
9,10,17977


# PHASE 3: DISTRICT MAPPING & DATA EXPORT

In [ ]:
print("🗺️ Mapping district names and exporting data...\n")

# 1. Create a dictionary with the names of Barcelona's districts
district_mapping = {
    1: 'Ciutat Vella',
    2: "L'Eixample",
    3: 'Sants-Montjuïc',
    4: 'Les Corts',
    5: 'Sarrià-Sant Gervasi',
    6: 'Gràcia',
    7: 'Horta-Guinardó',
    8: 'Nou Barris',
    9: 'Sant Andreu',
    10: 'Sant Martí'
}

# 2. Add the official names to the DataFrame using the map() function
df_renta_distritos['Nom_Districte'] = df_renta_distritos['Codi_Districte'].map(district_mapping)

# 3. Reorder columns for a cleaner, logical structure
df_renta_distritos = df_renta_distritos[['Codi_Districte', 'Nom_Districte', 'Renta_Media']]

# 4. Save the clean data to the 'processed' directory without the Pandas index
processed_dir = config.PROCESSED_DATA_DIR

os.makedirs(processed_dir, exist_ok=True)

output_path = os.path.join(processed_dir, '03_bcn_income_by_district.csv')

df_renta_distritos.to_csv(output_path, index=False)

print(f"✅ Success! Data formatted and saved at:")
print(f"📂 {output_path}")
print("\n🎉 EDA BCN INCOME BY DISTRICT COMPLETE.")

# 5. Display the final dataframe
display(df_renta_distritos)

🗺️ Mapping district names and exporting data...

✅ Success! Data formatted and saved at:
📂 /home/marvin/repos/pj-geo-yield-ai/data/processed/03_bcn_income_by_district.csv

🎉 EDA BCN INCOME BY DISTRICT COMPLETE.


,Codi_Districte,Nom_Districte,Renta_Media
0,1,Ciutat Vella,13990
1,2,Eixample,21976
2,3,Sants-Montjuïc,16911
3,4,Les Corts,26069
4,5,Sarrià-Sant Gervasi,31710
5,6,Gràcia,21251
6,7,Horta-Guinardó,17234
7,8,Nou Barris,13964
8,9,Sant Andreu,16979
9,10,Sant Martí,17977
